# Experiment 6: GloVe Word Embeddings and Statistical N-Gram Language Modeling

**Course**: Computational Linguistics and Natural Language Processing (CLNLP) Lab  
**Student Name**: Satwik Shukla  
**SAP ID**: 500123679  
**Batch**: 1 AIML | B.Tech CSE  

---

### Objectives:
1. Implement the GloVe (Global Vectors for Word Representation) algorithm to generate word embeddings from the given text corpus (`glove_corpus.txt`) and find the most similar words of "King":
   - Code 1: Without using any pretrained model (from scratch via co-occurrence matrix, weighting function, and AdaGrad optimization).
   - Code 2: Using a pretrained GloVe model (`glove-wiki-gigaword-50` via Gensim).
2. Implement a statistical language model from scratch using the N-gram approach (Bigram and Trigram) to predict the next word in a sequence based on preceding words in the specified corpus.


In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import re
import math
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import gensim.downloader as api


## Problem 1.1: GloVe Algorithm From Scratch (Without Pretrained Model)
We construct the word-word co-occurrence matrix $X$ using a symmetric sliding window with harmonic distance weighting $1/d$. 
We optimize the GloVe objective:
$$J = \sum_{i,j=1}^V f(X_{ij}) (w_i^T \tilde{w}_j + b_i + \tilde{b}_j - \log X_{ij})^2$$
where $f(x) = (x / x_{\max})^\alpha$ for $x < x_{\max}$ else $1.0$.


In [2]:
with open('glove_corpus.txt', 'r', encoding='utf-8') as f:
    raw_lines = f.readlines()

sentences = [re.sub(r'[^a-zA-Z\s]', '', line.lower()).split() for line in raw_lines if line.strip()]
vocab = sorted(list(set(word for sent in sentences for word in sent)))
w2i = {word: idx for idx, word in enumerate(vocab)}
i2w = {idx: word for idx, word in enumerate(vocab)}
V = len(vocab)

window_size = 4
cooc_dict = defaultdict(float)
for sent in sentences:
    for i, target in enumerate(sent):
        for dist in range(1, window_size + 1):
            if i - dist >= 0:
                ctx = sent[i - dist]
                cooc_dict[(w2i[target], w2i[ctx])] += 1.0 / dist
            if i + dist < len(sent):
                ctx = sent[i + dist]
                cooc_dict[(w2i[target], w2i[ctx])] += 1.0 / dist

pairs = list(cooc_dict.keys())
counts = np.array([cooc_dict[p] for p in pairs])
x_max = 10.0
alpha = 0.75
weights = np.where(counts < x_max, (counts / x_max) ** alpha, 1.0)
log_counts = np.log(counts)

embedding_dim = 30
np.random.seed(42)
W = (np.random.rand(V, embedding_dim) - 0.5) / embedding_dim
W_tilde = (np.random.rand(V, embedding_dim) - 0.5) / embedding_dim
b = np.zeros(V)
b_tilde = np.zeros(V)

gradsq_W = np.ones((V, embedding_dim))
gradsq_W_tilde = np.ones((V, embedding_dim))
gradsq_b = np.ones(V)
gradsq_b_tilde = np.ones(V)
learning_rate = 0.05
epochs = 150

for epoch in range(epochs):
    for idx, (i, j) in enumerate(pairs):
        f_val = weights[idx]
        diff = np.dot(W[i], W_tilde[j]) + b[i] + b_tilde[j] - log_counts[idx]
        loss_val = f_val * (diff ** 2)
        grad_common = 2.0 * f_val * diff
        grad_W = grad_common * W_tilde[j]
        grad_W_tilde = grad_common * W[i]
        grad_b = grad_common
        grad_b_tilde = grad_common
        
        W[i] -= (learning_rate / np.sqrt(gradsq_W[i])) * grad_W
        gradsq_W[i] += grad_W ** 2
        W_tilde[j] -= (learning_rate / np.sqrt(gradsq_W_tilde[j])) * grad_W_tilde
        gradsq_W_tilde[j] += grad_W_tilde ** 2
        b[i] -= (learning_rate / np.sqrt(gradsq_b[i])) * grad_b
        gradsq_b[i] += grad_b ** 2
        b_tilde[j] -= (learning_rate / np.sqrt(gradsq_b_tilde[j])) * grad_b_tilde
        gradsq_b_tilde[j] += grad_b_tilde ** 2

embeddings = W + W_tilde
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms[norms == 0] = 1e-10
normalized_embeddings = embeddings / norms

target_word = 'king'
target_idx = w2i[target_word]
sim_scores = np.dot(normalized_embeddings, normalized_embeddings[target_idx])
sorted_indices = np.argsort(-sim_scores)

results_scratch = []
for idx in sorted_indices:
    if idx != target_idx:
        results_scratch.append({'Word': i2w[idx], 'Cosine Similarity': round(float(sim_scores[idx]), 4)})
    if len(results_scratch) == 5:
        break

df_scratch = pd.DataFrame(results_scratch)
print('=== Top 5 Similar Words to "King" (GloVe From Scratch) ===')
print(df_scratch)


=== Top 5 Similar Words to "King" (GloVe From Scratch) ===
       Word  Cosine Similarity
0       the             0.9426
1     rules             0.8958
2     queen             0.7920
3        is             0.7494
4  princess             0.6927


## Problem 1.2: GloVe Word Embeddings Using Pretrained Gensim Model
We load the pretrained `glove-wiki-gigaword-50` vector set and retrieve the top nearest neighbors for "King".


In [3]:
pretrained_glove = api.load('glove-wiki-gigaword-50')
similar_words_gensim = pretrained_glove.most_similar('king', topn=5)

results_gensim = [{'Word': w, 'Cosine Similarity': round(float(score), 4)} for w, score in similar_words_gensim]
df_gensim = pd.DataFrame(results_gensim)
print('=== Top 5 Similar Words to "King" (Pretrained GloVe: glove-wiki-gigaword-50) ===')
print(df_gensim)


=== Top 5 Similar Words to "King" (Pretrained GloVe: glove-wiki-gigaword-50) ===
      Word  Cosine Similarity
0   prince             0.8236
1    queen             0.7839
2       ii             0.7746
3  emperor             0.7736
4      son             0.7667


## Problem 2: Statistical N-Gram Language Model From Scratch
We implement a statistical language model using Bigram and Trigram Maximum Likelihood Estimation (MLE) probabilities:
$$P(w_i | w_{i-1}) = \frac{C(w_{i-1}, w_i)}{C(w_{i-1})}$$
$$P(w_i | w_{i-2}, w_{i-1}) = \frac{C(w_{i-2}, w_{i-1}, w_i)}{C(w_{i-2}, w_{i-1})}$$


In [4]:
lm_corpus = """
natural language processing is a branch of artificial intelligence.
natural language processing enables computers to understand human language.
machine learning is a branch of artificial intelligence.
deep learning is a part of machine learning.
language models learn patterns from text data.
a language model predicts the next word in a sentence.
natural language processing uses language models.
artificial intelligence includes machine learning and deep learning.
"""

sentences_lm = [re.sub(r'[^a-zA-Z\s]', '', line.lower()).split() for line in lm_corpus.strip().split('\n') if line.strip()]

unigrams = Counter()
bigrams = Counter()
trigrams = Counter()

for sent in sentences_lm:
    padded = ['<s>'] + sent + ['</s>']
    for w in padded:
        unigrams[w] += 1
    for i in range(len(padded) - 1):
        bigrams[(padded[i], padded[i+1])] += 1
    for i in range(len(padded) - 2):
        trigrams[(padded[i], padded[i+1], padded[i+2])] += 1

def predict_next_word(context_str, n=2, top_k=3):
    tokens = context_str.strip().lower().split()
    candidates = {}
    if n == 2:
        prev_word = tokens[-1]
        denom = unigrams[prev_word]
        if denom > 0:
            for pair, count in bigrams.items():
                if pair[0] == prev_word and pair[1] != '</s>':
                    candidates[pair[1]] = count / denom
    elif n == 3:
        if len(tokens) >= 2:
            prev_context = (tokens[-2], tokens[-1])
            denom = bigrams[prev_context]
            if denom > 0:
                for tri, count in trigrams.items():
                    if (tri[0], tri[1]) == prev_context and tri[2] != '</s>':
                        candidates[tri[2]] = count / denom
        if not candidates and len(tokens) >= 1:
            prev_word = tokens[-1]
            denom = unigrams[prev_word]
            if denom > 0:
                for pair, count in bigrams.items():
                    if pair[0] == prev_word and pair[1] != '</s>':
                        candidates[pair[1]] = count / denom
    sorted_items = sorted(candidates.items(), key=lambda x: -x[1])[:top_k]
    return [{'Predicted Next Word': w, 'Conditional Probability': round(float(prob), 4)} for w, prob in sorted_items]

test_prompts = [
    ('natural language', 3),
    ('machine', 2),
    ('deep', 2),
    ('artificial', 2),
    ('language', 2)
]

prediction_records = []
for prompt, n_val in test_prompts:
    preds = predict_next_word(prompt, n=n_val, top_k=1)
    if preds:
        prediction_records.append({
            'Input Context': prompt,
            'Model Type': f'{n_val}-Gram',
            'Predicted Word': preds[0]['Predicted Next Word'],
            'Probability': preds[0]['Conditional Probability']
        })

df_predictions = pd.DataFrame(prediction_records)
print('=== Statistical N-Gram Language Model Predictions ===')
print(df_predictions)


=== Statistical N-Gram Language Model Predictions ===
      Input Context Model Type Predicted Word  Probability
0  natural language     3-Gram     processing       1.0000
1           machine     2-Gram       learning       1.0000
2              deep     2-Gram       learning       1.0000
3        artificial     2-Gram   intelligence       1.0000
4          language     2-Gram     processing       0.4286
